- Write PySpark Code to write desired output : 2nd DAY_SALE is the sale after the repeat sale (ex- 1st Sell's 2nd DAY SALE is the third sale)

- Identify the sale_date which has made 3rd highest sale for each product.

In [0]:
from pyspark.sql.functions import lead, lag, col, to_date, desc, asc, rank, dense_rank, row_number, split, explode, collect_list, unix_timestamp, count, round,coalesce,lit
from pyspark.sql.window import Window

In [0]:
data = [('TV','2016-11-27',800),
        ('TV','2016-11-30',900),
        ('TV','2016-12-29',500),
        ('TV','2017-11-20',400),
        ('FRIDGE','2016-10-11',760),
        ('FRIDGE','2016-10-13',400),
        ('FRIDGE','2016-11-27',460)
    ]

schema = ["product","sale_date","amount"]

df = spark.createDataFrame(data,schema).withColumn("sale_date",to_date(col("sale_date"),"yyyy-MM-dd"))
display(df)

In [0]:
window = Window.partitionBy("product").orderBy(col("sale_date"))

df_final = df.withColumn("NEXT_2NDDAY_SALE",lead("amount",2).over(window)).withColumn("PREVIOUS_2NDDAY_SALE",lag("amount",2).over(window))

df_final2 = df_final.withColumn("diff",coalesce((col("NEXT_2NDDAY_SALE")-col("PREVIOUS_2NDDAY_SALE")), lit(0)))
display(df_final2)

- Identify the sale_date which has made 3rd highest sale for each product.

In [0]:
data = [('TV','2016-11-27',800),
        ('TV','2016-11-30',900),
        ('TV','2016-12-29',500),
        ('TV','2017-11-20',400),
        ('FRIDGE','2016-10-11',760),
        ('FRIDGE','2016-10-13',400),
        ('FRIDGE','2016-11-27',460)
    ]

schema = ["product","sale_date","amount"]

df = spark.createDataFrame(data,schema).withColumn("sale_date",to_date(col("sale_date"),"yyyy-MM-dd"))
display(df)

In [0]:
window = Window.partitionBy("product").orderBy(col("amount").desc())

df_sales = df.withColumn("rank",dense_rank().over(window)).where("rank = 3").select("product","sale_date","amount")
display(df_sales)

**Question :** Find the question_id with the highest answer rate from the survey_log table
- Answer rate is ratio of answer actions to show actions for that question (only count show events in the denominator) if two answer ratio is same return the question id which is less

In [0]:
%sql
CREATE OR REPLACE TABLE survey_log (
  id INT,
  action STRING,
  question_id INT,
  answer_id INT,
  q_num INT,
  timestamp INT
) USING DELTA;

%sql
INSERT INTO survey_log VALUES
(5, 'show', 285, NULL, 1, 123),
(5, 'answer', 285, 124124, 1, 124),
(5, 'show', 369, NULL, 2, 125),
(5, 'skip', 369, NULL, 2, 126),

-- Additional rows for more questions/users
(6, 'show', 285, NULL, 1, 130),
(6, 'show', 369, NULL, 1, 131),
-- (6, 'answer', 369, 456789, 1, 132),
(7, 'show', 285, NULL, 1, 140),
(7, 'show', 369, NULL, 2, 141),
(7, 'skip', 285, NULL, 1, 142);

In [0]:
%sql
SELECT * FROM survey_log ORDER BY id, timestamp;

In [0]:
from pyspark.sql import functions as F

In [0]:
df_survey = spark.sql("select * from survey_log")

In [0]:
%sql
SELECT question_id, count(case when action = "answer" then 1 end)/count(case when action = 'show' then 1 end)
as answer_rate_ratio
FROM survey_log
group by question_id
ORDER BY answer_rate_ratio DESC
limit 1

**Approach 2**

In [0]:
%sql

select sum(case when action = 'answer' then 1 else 0 end) over (partition by (question_id))/sum(case when action = 'show' then 1 else 0 end) over (partition by (question_id)) as ansratio, *
from survey_log
order by ansratio desc limit 1

**Approach 3**

In [0]:
window = Window.partitionBy("question_id")
df_survey_final = df_survey.withColumn("ansRatio", F.sum(F.when(df_survey.action == 'answer', 1).otherwise(0)).over(window)/F.sum(F.when(df_survey.action == 'show', 1).otherwise(0)).over(window)).orderBy(F.col("ansRatio").desc()).limit(1)

display(df_survey_final)
                                       

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
data = [
    ('A','X',80),
    ('A','Y',70),
    ('A','Z',75),
    ('B','X',90),
    ('B','Y',91),
    ('B','Z',75),
    ('C','X',60),
    ('C','Y',93),
    ('C','Z',81)
]

schema = ["StudentID","SubjectID","MarksObtained"]

df_student = spark.createDataFrame(data,schema)
display(df_student)

- Display the students whose total of two highest marks that are atleast 160

In [0]:
window = Window.partitionBy("StudentID").orderBy(F.col("MarksObtained").desc())

df = df_student.withColumn("rank", row_number().over(window)).where("rank <=2")

final_df = df.groupBy("StudentID").agg(F.sum(col("MarksObtained")).alias("SumOfMarks")).where("SumOfMarks>=160")

display(final_df)

In [0]:
%sql
create table family 
(
person varchar(5),
type varchar(10),
age int
);

insert into family values ('A1','Adult',54)
,('A2','Adult',53),('A3','Adult',52),('A4','Adult',58),('A5','Adult',54),('C1','Child',20),('C2','Child',19),('C3','Child',22),('C4','Child',15);

In [0]:
%sql
select * from family;

In [0]:
%sql
with adult_family as (
    select *,row_number() over(order by person) as rn 
    from family where type = "Adult"
),
child_family as (
    select *,row_number() over(order by person) as rn 
    from family where type = "Child"
)
select a.person,c.person from adult_family a left join child_family c on a.rn = c.rn; 

In [0]:
%sql
with adult_family as (
    select *,row_number() over(order by age desc) as rn 
    from family where type = "Adult"
),
child_family as (
    select *,row_number() over(order by age) as rn 
    from family where type = "Child"
)
select a.person,c.person, a.age, c.age from adult_family a left join child_family c on a.rn = c.rn; 